# Variance-controlled arm: violatoronly (silly-only, 20 seeds) -- PARALLEL half

One arm of the CRN-paired pair; run this on its own GPU alongside the other arm's notebook. Both use 20 seeds so the keys match and the pair stays CRN-coupled. When both CSVs exist, run `vc_analysis.report(...)` (or `python vc_analysis.py <vo.csv> <eo.csv> 0.02`) on either machine.

## 0. Setup (pure Python; leaves the pod's jax untouched)

In [ ]:
# jax is already installed & CUDA-bound on the pod -- do NOT reinstall/upgrade it.
# Only add flax/optax if missing, BEFORE importing jax (so no fork with jax threads live).
try:
    import flax, optax
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flax', 'optax'])
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
import jax
print('jax', jax.__version__, '| devices:', jax.devices())
if jax.devices()[0].platform != 'gpu':
    print('WARNING: NOT on GPU -> run_extinction will refuse a real run. Rebind & re-run.')


## 1. Write sources (single cell: base64 -> files)

In [ ]:
import base64, pathlib
_SRC = {
  'berryworld.py': '...',
  'berryworld_jax.py': '...',
  'train_jax.py': '...',
  'run_sweep.py': 'IiIiCnJ1bl9zd2VlcC5weSAtLSB0aGUgZmFpdGhmdWwtc2NhbGUgcG9wdWxhdGlvbiBzd2VlcCwgZHJpdmVyIGZvciBHUFUgKENvbGFiKS4KCkZvciBlYWNoIChOLCBjb25kaXRpb24pIGNlbGwgaXQgdm1hcHMgdGhlIEpBWCBJUFBPIHRyYWluIG92ZXIgc2VlZHMgYW5kIGxvZ3MKcGVyLXVwZGF0ZSBtZXRyaWNzLiBUaGUgcXVlc3Rpb24gdGhpcyBhbnN3ZXJzOiBkb2VzIGVuZm9yY2VtZW50IChzZWxlY3Rpdml0eT4xKQpmaXJlIHdoZW4gdGhlIHBvcHVsYXRpb24gZ3Jvd3MsIGdpdmVuIGVub3VnaCB0cmFpbmluZz8KCkxvY2FsIChDUFUpIHNtb2tlOiAgcHl0aG9uIHJ1bl9zd2VlcC5weSAtLXNtb2tlCkZhaXRoZnVsIChHUFUpOiAgICAgaW1wb3J0ZWQgZnJvbSB0aGUgQ29sYWIgbm90ZWJvb2sgd2l0aCBhIGxhcmdlIGhwLgoKTk9URSAvIGhvbmVzdCBsaW1pdGF0aW9ucyBvZiB0aGlzIGRpYWdub3N0aWMgdmVyc2lvbjoKICAqIEFsbCBOIGFnZW50cyBwbGF5IGV2ZXJ5IGVwaXNvZGUgKEtvc3RlcidzIDgtb2YtMTIgcGVyLWVwaXNvZGUgc2FtcGxpbmcgaXMgYQogICAgcmVmaW5lbWVudCBub3QgeWV0IGltcGxlbWVudGVkIC0tIHRoaXMgdGVzdHMgcG9wdWxhdGlvbiBTSVpFLCB0aGUgcHJpbWFyeQogICAgbGV2ZXIpLgogICogcGF0Y2hfbWFzayBpcyBmaXhlZCBwZXIgKE4sY29uZGl0aW9uKSBjZWxsIGFjcm9zcyBzZWVkczsgc2VlZCB2YXJpZXMgbmV0CiAgICBpbml0ICsgcm9sbG91dCArIHJlc2V0cy4gUGVyLXNlZWQgcGF0Y2hlcyBhcmUgdGhlIHJpZ29yb3VzIHZlcnNpb24sIGxhdGVyLgoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IGpheAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRyYWluX2pheCBhcyBUCmltcG9ydCBiZXJyeXdvcmxkX2pheCBhcyBid2oKCgpkZWYgZW52X3ZhcmlhbnQocG9pc29uX2RlbGF5PTI1LCByX3phcF9ib251cz0wLjAsCiAgICAgICAgICAgICAgICBlcGlzb2RlX2xlbj0zMDAsIHphcF9yZW1vdmFsX3N0ZXBzPTI1LCBvYnNlcnZlX3BlbmRpbmc9RmFsc2UsCiAgICAgICAgICAgICAgICBib251c19yZXF1aXJlc19tYXJrPUZhbHNlLCBhdXRvX3RhcmdldD1GYWxzZSwgY196YXBwZWQ9Mi4wLAogICAgICAgICAgICAgICAgbl9iZXJyeV90eXBlcz0yLCBncmlkPTE1LCBnaG9zdF9rZWVwc19ib251cz1UcnVlKToKICAgIHJldHVybiBkaWN0KHBvaXNvbl9kZWxheT1wb2lzb25fZGVsYXksIHJfemFwX2JvbnVzPXJfemFwX2JvbnVzLAogICAgICAgICAgICAgICAgZXBpc29kZV9sZW49ZXBpc29kZV9sZW4sIHphcF9yZW1vdmFsX3N0ZXBzPXphcF9yZW1vdmFsX3N0ZXBzLAogICAgICAgICAgICAgICAgb2JzZXJ2ZV9wZW5kaW5nPW9ic2VydmVfcGVuZGluZywKICAgICAgICAgICAgICAgIGJvbnVzX3JlcXVpcmVzX21hcms9Ym9udXNfcmVxdWlyZXNfbWFyaywKICAgICAgICAgICAgICAgIGF1dG9fdGFyZ2V0PWF1dG9fdGFyZ2V0LCBjX3phcHBlZD1jX3phcHBlZCwKICAgICAgICAgICAgICAgIG5fYmVycnlfdHlwZXM9bl9iZXJyeV90eXBlcywgZ3JpZD1ncmlkLAogICAgICAgICAgICAgICAgZ2hvc3Rfa2VlcHNfYm9udXM9Z2hvc3Rfa2VlcHNfYm9udXMpCgoKZGVmIHJ1bl9zd2VlcChuX2xpc3QsIGNvbmRpdGlvbnMsIG5fc2VlZHMsIGhwLCBlbnZfbGlzdCwgb3V0X2Nzdj0ic3dlZXAuY3N2IiwKICAgICAgICAgICAgICB2bWFwX3NlZWRzPUZhbHNlKToKICAgICIiIlN3ZWVwIG92ZXIgTiB4IGVudl92YXJpYW50IHggY29uZGl0aW9uLiBFYWNoIGVudl92YXJpYW50IGNhcnJpZXMgaXRzIG93bgogICAgcG9pc29uX2RlbGF5IChkaWZmaWN1bHR5KSBhbmQgcl96YXBfYm9udXMgKGVuZm9yY2VtZW50IGluY2VudGl2ZSksIGJvdGgKICAgIGxvZ2dlZCBwZXIgcm93IHNvIHRoZSBEIGFuZCBib251cyBheGVzIGFyZSBhbmFseXNhYmxlLiIiIgogICAgaWYgaXNpbnN0YW5jZShlbnZfbGlzdCwgZGljdCk6ICAgICAgICAgICAgICAgICAjIGFsbG93IGEgc2luZ2xlIGVudgogICAgICAgIGVudl9saXN0ID0gW2Vudl9saXN0XQogICAgIyBHdWFyZDogcmVmdXNlIGEgcmVhbCBzd2VlcCBvbiBDUFUuIFRoZSB3aG9sZSBwb2ludCBvZiB0aGUgSkFYIHBpdm90IGlzCiAgICAjIEdQVTsgYSBDUFUgcnVuIGhlcmUgbWVhbnMgdGhlIG5vdGVib29rIGlzIGV4ZWN1dGluZyBvbiB0aGUgbGFwdG9wLCBub3QKICAgICMgQ29sYWIuIEFib3J0IGluIDwxcyBpbnN0ZWFkIG9mIGdyaW5kaW5nIGZvciBob3Vycy4gVGlueSBkZXZpY2Utc2FuaXR5CiAgICAjIHJ1bnMgKHVwZGF0ZXMqbnVtX2VudnMgPD0gMTAwMCkgYXJlIHN0aWxsIGFsbG93ZWQgb24gQ1BVLgogICAgaWYgbm90IGFueShkLnBsYXRmb3JtID09ICJncHUiIGZvciBkIGluIGpheC5kZXZpY2VzKCkpIFwKICAgICAgICAgICAgYW5kIGhwWyJ1cGRhdGVzIl0gKiBocFsibnVtX2VudnMiXSA+IDEwMDA6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICBmInJ1bl9zd2VlcCBhYm9ydGVkOiBOTyBHUFUgKGpheC5kZXZpY2VzKCk9e2pheC5kZXZpY2VzKCl9KS4gIgogICAgICAgICAgICBmIlRoaXMgaXMgYSBDUFUgcnVudGltZSAtLSB5b3VyIGxhcHRvcCwgbm90IENvbGFiIEdQVSAtLSBhbmQgIgogICAgICAgICAgICBmInVwZGF0ZXMqbnVtX2VudnM9e2hwWyd1cGRhdGVzJ10gKiBocFsnbnVtX2VudnMnXX0gaXMgYSByZWFsICIKICAgICAgICAgICAgZiJzd2VlcCAoaG91cnMgb24gQ1BVKS4gT24gQ29sYWI6IFJ1bnRpbWUgLT4gQ2hhbmdlIHJ1bnRpbWUgdHlwZSAiCiAgICAgICAgICAgIGYiLT4gR1BVLCB0aGVuIFJ1biBhbGwuIikKICAgICMgQ3Jhc2gtc2FmZTogd3JpdGUgZWFjaCBjZWxsJ3Mgcm93cyB0aGUgaW5zdGFudCBpdCBmaW5pc2hlcyBhbmQgZmx1c2gsIHNvIGFuCiAgICAjIGludGVycnVwdCAvIENvbGFiIHRpbWVvdXQgLyBwcmVlbXB0aW9uIGtlZXBzIGV2ZXJ5IENPTVBMRVRFRCBjZWxsIGluc3RlYWQgb2YKICAgICMgbG9zaW5nIHRoZSB3aG9sZSBydW4gKHRoZSBlYXJsaWVyIGZhaWx1cmUgbW9kZSAtLSBzZWUgcHJvamVjdCBub3RlcyBvbgogICAgIyBtYWtpbmcgb3V0cHV0cyByZXN1bWFibGUpLiBgcm93c2AgaXMgc3RpbGwgYWNjdW11bGF0ZWQgZm9yIHRoZSByZXR1cm4gdmFsdWUuCiAgICBuYnQgPSBlbnZfbGlzdFswXS5nZXQoIm5fYmVycnlfdHlwZXMiLCAyKSAgICMgcGVyLXR5cGUgZWF0L2VuYyBjb2x1bW5zCiAgICBmaWVsZG5hbWVzID0gKFsiTiIsICJEIiwgImJvbnVzIiwgImNvbmRpdGlvbiIsICJzZWVkIiwgInVwZGF0ZSJdCiAgICAgICAgICAgICAgICAgICsgW2YiZWF0e3R9IiBmb3IgdCBpbiByYW5nZShuYnQpXSArIFtmImVuY3t0fSIgZm9yIHQgaW4gcmFuZ2UobmJ0KV0KICAgICAgICAgICAgICAgICAgKyBbInNlbGVjdGl2aXR5IiwgInByZXZhbGVuY2UiLCAiemFwcyIsICJyZXQiXSkKICAgIHJvd3MgPSBbXQogICAgIyBSZXByb2R1Y2liaWxpdHk6IGR1bXAgdGhlIEZVTEwgcmVjaXBlIG5leHQgdG8gdGhlIENTVi4gVGhlIHBpbG90IHByb3ZlZCB0aGUKICAgICMgQ1NWJ3MgNCBjb25maWcgY29sdW1ucyAoTi9EL2JvbnVzL3NlZWQpIFVOREVSLXNwZWNpZnkgYSBydW4gLS0gbnVtX2VudnMgd2FzCiAgICAjIHRoZSBkaWZmZXJlbmNlIHRoYXQgZmxpcHBlZCB0aGUgd2hvbGUgcmVzdWx0IC0tIHNvIHBlcnNpc3QgaHAgKyBlbnYgKyBzZWVkcy4KICAgIHdpdGggb3BlbihvdXRfY3N2ICsgIi5jb25maWcuanNvbiIsICJ3IikgYXMgY2Y6CiAgICAgICAganNvbi5kdW1wKHsiaHAiOiBocCwgIm5fc2VlZHMiOiBuX3NlZWRzLAogICAgICAgICAgICAgICAgICAgImNvbmRpdGlvbnMiOiBbbGlzdChjKSBmb3IgYyBpbiBjb25kaXRpb25zXSwKICAgICAgICAgICAgICAgICAgICJlbnZfbGlzdCI6IFtkaWN0KGUpIGZvciBlIGluIGVudl9saXN0XSwKICAgICAgICAgICAgICAgICAgICJuX2xpc3QiOiBsaXN0KG5fbGlzdCl9LCBjZiwgaW5kZW50PTIpCiAgICBmID0gb3BlbihvdXRfY3N2LCAidyIsIG5ld2xpbmU9IiIpCiAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1maWVsZG5hbWVzKQogICAgdy53cml0ZWhlYWRlcigpOyBmLmZsdXNoKCkKICAgIHRyeToKICAgICAgICBmb3IgTiBpbiBuX2xpc3Q6CiAgICAgICAgICAgIGZvciBlbnYgaW4gZW52X2xpc3Q6CiAgICAgICAgICAgICAgICBmb3IgbWFya2VkIGluIGNvbmRpdGlvbnM6CiAgICAgICAgICAgICAgICAgICAgY2ZnID0gYndqLkpDZmcoCiAgICAgICAgICAgICAgICAgICAgICAgIG5fYWdlbnRzPU4sIGVwaXNvZGVfbGVuPWVudlsiZXBpc29kZV9sZW4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgcG9pc29uX2RlbGF5PWVudlsicG9pc29uX2RlbGF5Il0sCiAgICAgICAgICAgICAgICAgICAgICAgIHphcF9yZW1vdmFsX3N0ZXBzPWVudlsiemFwX3JlbW92YWxfc3RlcHMiXSwKICAgICAgICAgICAgICAgICAgICAgICAgcl96YXBfYm9udXM9ZW52WyJyX3phcF9ib251cyJdLAogICAgICAgICAgICAgICAgICAgICAgICBvYnNlcnZlX3BlbmRpbmc9ZW52LmdldCgib2JzZXJ2ZV9wZW5kaW5nIiwgRmFsc2UpLAogICAgICAgICAgICAgICAgICAgICAgICBib251c19yZXF1aXJlc19tYXJrPWVudi5nZXQoImJvbnVzX3JlcXVpcmVzX21hcmsiLCBGYWxzZSksCiAgICAgICAgICAgICAgICAgICAgICAgIGF1dG9fdGFyZ2V0PWVudi5nZXQoImF1dG9fdGFyZ2V0IiwgRmFsc2UpLAogICAgICAgICAgICAgICAgICAgICAgICBjX3phcHBlZD1lbnYuZ2V0KCJjX3phcHBlZCIsIDIuMCksCiAgICAgICAgICAgICAgICAgICAgICAgIGdyaWQ9ZW52LmdldCgiZ3JpZCIsIDE1KSwKICAgICAgICAgICAgICAgICAgICAgICAgbl9iZXJyeV90eXBlcz1lbnYuZ2V0KCJuX2JlcnJ5X3R5cGVzIiwgMiksCiAgICAgICAgICAgICAgICAgICAgICAgIG1hcmtlZF9tYXNrPXR1cGxlKHQgaW4gbWFya2VkIGZvciB0IGluIHJhbmdlKGVudi5nZXQoIm5fYmVycnlfdHlwZXMiLCAyKSkpKQogICAgICAgICAgICAgICAgICAgIHBtID0gVC5idWlsZF9wYXRjaF9tYXNrKG1hcmtlZCwgTiwgbl9iZXJyeV90eXBlcz1lbnYuZ2V0KCJuX2JlcnJ5X3R5cGVzIiwgMiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZ3JpZD1lbnYuZ2V0KCJncmlkIiwgMTUpKQogICAgICAgICAgICAgICAgICAgIHRyYWluMSA9IFQubWFrZV90cmFpbihjZmcsIHBtLCBocCkKICAgICAgICAgICAgICAgICAgICBrZXlzID0gamF4LnJhbmRvbS5zcGxpdChqYXgucmFuZG9tLlBSTkdLZXkoMCksIG5fc2VlZHMpCiAgICAgICAgICAgICAgICAgICAgIyBTZWVkcyBhcmUgSU5ERVBFTkRFTlQgKG5vIGNyb3NzLXNlZWQgcmVkdWN0aW9uKSwgc28gdm1hcHBpbmcKICAgICAgICAgICAgICAgICAgICAjIHRoZW0gb25seSBwYXJhbGxlbGlzZXMgLS0gaXQgZG9lcyBub3QgY2hhbmdlIHBlci1zZWVkIG1hdGgsCiAgICAgICAgICAgICAgICAgICAgIyBidXQgaXQgaG9sZHMgYWxsIG5fc2VlZHMgdHJhaW5pbmcgZ3JhcGhzIG9uIHRoZSBHUFUgYXQgb25jZQogICAgICAgICAgICAgICAgICAgICMgKH5uX3NlZWRzIHggbWVtb3J5LCB0aGUgT09NIGF0IGxhcmdlIG51bV9lbnZzKS4gRGVmYXVsdCBydW5zCiAgICAgICAgICAgICAgICAgICAgIyB0aGVtIFNFUVVFTlRJQUxMWTogb25lIHNlZWQncyBncmFwaCByZXNpZGVudCBhdCBhIHRpbWUsIGVhY2gKICAgICAgICAgICAgICAgICAgICAjIHB1bGxlZCB0byBob3N0IHNvIGl0cyBkZXZpY2UgYnVmZmVycyBmcmVlIGJlZm9yZSB0aGUgbmV4dC4KICAgICAgICAgICAgICAgICAgICAjIHZtYXBfc2VlZHM9VHJ1ZSByZXN0b3JlcyB0aGUgcGFyYWxsZWwgcGF0aCAobmVlZHMgdGhlIFZSQU07CiAgICAgICAgICAgICAgICAgICAgIyBpdHMgWExBIGNvZGVnZW4gZGlmZmVycywgc28ga2VlcCBpdCBmb3IgZXhhY3Qta29zdGVyIHJlcHJvKS4KICAgICAgICAgICAgICAgICAgICBpZiB2bWFwX3NlZWRzOgogICAgICAgICAgICAgICAgICAgICAgICBtZm4gPSBqYXguaml0KGpheC52bWFwKGxhbWJkYSBrOiB0cmFpbjEoaylbMV0pKQogICAgICAgICAgICAgICAgICAgICAgICBtbSA9IGpheC5ibG9ja191bnRpbF9yZWFkeShtZm4oa2V5cykpICAgICAgICAgICMgZWFjaCAoUywgVSkKICAgICAgICAgICAgICAgICAgICAgICAgbSA9IHtrazogbnAuYXNhcnJheSh2dikgZm9yIGtrLCB2diBpbiBtbS5pdGVtcygpfQogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHNlZWRfbWV0cmljcyA9IGpheC5qaXQobGFtYmRhIGs6IHRyYWluMShrKVsxXSkKICAgICAgICAgICAgICAgICAgICAgICAgcGVyX3NlZWQgPSBbXQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgayBpbiBrZXlzOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgbWsgPSBqYXguYmxvY2tfdW50aWxfcmVhZHkoc2VlZF9tZXRyaWNzKGspKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcGVyX3NlZWQuYXBwZW5kKHtrazogbnAuYXNhcnJheSh2dikKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGtrLCB2diBpbiBtay5pdGVtcygpfSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRlbCBtawogICAgICAgICAgICAgICAgICAgICAgICBtID0ge2trOiBucC5zdGFjayhbcHNba2tdIGZvciBwcyBpbiBwZXJfc2VlZF0sIDApICAjIChTLCBVKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBrayBpbiBwZXJfc2VlZFswXX0KICAgICAgICAgICAgICAgICAgICBjb25kID0gIiIuam9pbihzdHIoYikgZm9yIGIgaW4gbWFya2VkKSBvciAibm9uZSIKICAgICAgICAgICAgICAgICAgICBVID0gbVsiZWF0MCJdLnNoYXBlWzFdCiAgICAgICAgICAgICAgICAgICAgZSA9IG1bImVhdDAiXTsgc2VsID0gbVsic2VsZWN0aXZpdHkiXQogICAgICAgICAgICAgICAgICAgIHByaW50KGYiTj17TjoyZH0gRD17ZW52Wydwb2lzb25fZGVsYXknXToyZH0gIgogICAgICAgICAgICAgICAgICAgICAgICAgIGYiYm9udXM9e2Vudlsncl96YXBfYm9udXMnXTouMWZ9IHtjb25kOjRzfSAgIgogICAgICAgICAgICAgICAgICAgICAgICAgIGYiZWF0MCB7ZVs6LCA6NV0ubWVhbigpOjQuMGZ9LT57ZVs6LCAtNTpdLm1lYW4oKTo0LjBmfSIKICAgICAgICAgICAgICAgICAgICAgICAgICBmIiAgc2VsIHtzZWxbOiwgLTU6XS5tZWFuKCk6LjJmfSAgKG49e25fc2VlZHN9KSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICAgICAgY2VsbF9yb3dzID0gW2RpY3QoCiAgICAgICAgICAgICAgICAgICAgICAgIE49TiwgRD1lbnZbInBvaXNvbl9kZWxheSJdLAogICAgICAgICAgICAgICAgICAgICAgICBib251cz1lbnZbInJfemFwX2JvbnVzIl0sIGNvbmRpdGlvbj1jb25kLAogICAgICAgICAgICAgICAgICAgICAgICBzZWVkPXMsIHVwZGF0ZT11LAogICAgICAgICAgICAgICAgICAgICAgICBzZWxlY3Rpdml0eT1mbG9hdChtWyJzZWxlY3Rpdml0eSJdW3MsIHVdKSwKICAgICAgICAgICAgICAgICAgICAgICAgcHJldmFsZW5jZT1mbG9hdChtWyJwcmV2YWxlbmNlIl1bcywgdV0pLAogICAgICAgICAgICAgICAgICAgICAgICB6YXBzPWZsb2F0KG1bInphcHMiXVtzLCB1XSksIHJldD1mbG9hdChtWyJyZXQiXVtzLCB1XSksCiAgICAgICAgICAgICAgICAgICAgICAgICoqe2YiZWF0e3R9IjogZmxvYXQobVtmImVhdHt0fSJdW3MsIHVdKSBmb3IgdCBpbiByYW5nZShuYnQpfSwKICAgICAgICAgICAgICAgICAgICAgICAgKip7ZiJlbmN7dH0iOiBmbG9hdChtW2YiZW5je3R9Il1bcywgdV0pIGZvciB0IGluIHJhbmdlKG5idCl9KQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgcyBpbiByYW5nZShuX3NlZWRzKSBmb3IgdSBpbiByYW5nZShVKV0KICAgICAgICAgICAgICAgICAgICB3LndyaXRlcm93cyhjZWxsX3Jvd3MpOyBmLmZsdXNoKCkgICAgICAgICAgIyBwZXJzaXN0IHRoaXMgY2VsbAogICAgICAgICAgICAgICAgICAgIHJvd3MuZXh0ZW5kKGNlbGxfcm93cykKICAgIGZpbmFsbHk6CiAgICAgICAgZi5jbG9zZSgpCiAgICBwcmludChmIndyb3RlIHtsZW4ocm93cyl9IHJvd3MgLT4ge291dF9jc3Z9IiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiByb3dzCgoKRU5WID0gZW52X3ZhcmlhbnQoKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBEPTI1LCBib251cz0wIGRlZmF1bHQKCiMgRmFpdGhmdWwtc2NhbGUgaHlwZXJwYXJhbWV0ZXJzIGZvciBHUFUuIHN0ZXBzID0gdXBkYXRlcyAqIG51bV9lbnZzICogZXBpc29kZV9sZW4uCiMgdXBkYXRlcz0xNTAwLCBudW1fZW52cz0yNTYgLT4gfjEuMTVlOCBzdGVwcy9ydW4gKEtvc3RlciByZWdpbWUgaXMgMi00ZTgpLgpGQUlUSEZVTF9IUCA9IGRpY3QoaGlkZGVuPTY0LCBscj0zZS00LCBnYW1tYT0wLjk5LCBsYW09MC45NSwgY2xpcD0wLjIsCiAgICAgICAgICAgICAgICAgICBlcG9jaHM9MywgZW50PTAuMDEsIHZmPTAuNSwgbWF4X2dyYWQ9MC41LAogICAgICAgICAgICAgICAgICAgbnVtX2VudnM9MjU2LCB1cGRhdGVzPTE1MDApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNtb2tlIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0idGlueSBDUFUgcnVuIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKICAgIGlmIGEuc21va2U6CiAgICAgICAgaHAgPSBkaWN0KEZBSVRIRlVMX0hQKTsgaHBbIm51bV9lbnZzIl0gPSA4OyBocFsidXBkYXRlcyJdID0gMjAKICAgICAgICBlbnZzID0gW2Vudl92YXJpYW50KHBvaXNvbl9kZWxheT1kKSBmb3IgZCBpbiAoMjUsIDc1KV0gICAgICMgRCBheGlzCiAgICAgICAgcnVuX3N3ZWVwKFsxMl0sIFsoKSwgKDAsKSwgKDAsIDEpXSwgbl9zZWVkcz0yLCBocD1ocCwgZW52X2xpc3Q9ZW52cywKICAgICAgICAgICAgICAgICAgb3V0X2Nzdj0ic3dlZXBfc21va2UuY3N2IikKICAgIGVsc2U6CiAgICAgICAgcnVuX3N3ZWVwKFsxMCwgMTJdLCBbKCksICgwLCksICgwLCAxKV0sIG5fc2VlZHM9OCwKICAgICAgICAgICAgICAgICAgaHA9RkFJVEhGVUxfSFAsIGVudl9saXN0PVtFTlZdLCBvdXRfY3N2PSJzd2VlcC5jc3YiKQo=',
  'extinct.py': '...',
  'vc_analysis.py': 'IiIiVmFyaWFuY2UtY29udHJvbGxlZCB2aW9sYXRvci1vbmx5IGFuYWx5c2lzOiBlcXVpdmFsZW5jZSBib3VuZCBvbiB0aGUgdmlvbGF0b3Itb25seQpzaWxseSBudWxsICsgQ1JOLXBhaXJlZCBpbnRlcmFjdGlvbiBhZ2FpbnN0IHRoZSBlbmZvcmNlci1vbmx5IGFybS4KClRoZSAyeDIgaGVhZGxpbmUgaXMgYW4gSU5URVJBQ1RJT046IGdhdGluZyB0aGUgdmlvbGF0b3IncyBjb3N0IGFsb25lIGRvZXMgbm90aGluZwooc2lsbHkgaG9sZHMpIHdoaWxlIGdhdGluZyB0aGUgZW5mb3JjZXIncyBpbmNlbnRpdmUgYWxvbmUgZGlzc29sdmVzIHRoZSBub3JtIChzaWxseQpkZWNheXMgYXMgdW5kZXIgZnVsbCByZW1vdmFsKS4gQSBiYXJlIG51bGwgaXMgd2VhazsgdGhpcyBxdWFudGlmaWVzIGl0IHR3byB3YXlzOgoKICAoMSkgRVFVSVZBTEVOQ0UgKFRPU1QpIG9uIHRoZSB2aW9sYXRvci1vbmx5IHNpbGx5IHJlbF9kZWNheSAtLSBpcyBpdCBzdGF0aXN0aWNhbGx5CiAgICAgIGluc2lkZSBhIHByZS1zZXQgbmVnbGlnaWJsZSBiYW5kICsvLWVwcyAoaS5lLiBhICpzdXBwb3J0ZWQqIG51bGwsIG5vdCBqdXN0IG4ucy4pPwogICgyKSBQQUlSRUQgSU5URVJBQ1RJT04gLS0gcGVyLXNlZWQgKGVuZm9yY2VyLW9ubHkgZGVjYXkgLSB2aW9sYXRvci1vbmx5IGRlY2F5KS4KICAgICAgVmFsaWQgT05MWSBpZiBib3RoIGFybXMgcmFuIGF0IHRoZSBzYW1lIG5fc2VlZHMgaW4gdGhlIHNhbWUgc2Vzc2lvbiwgc28KICAgICAgamF4LnJhbmRvbS5zcGxpdChQUk5HS2V5KDApLCBuX3NlZWRzKSB5aWVsZHMgaWRlbnRpY2FsIGtleXMgPT4gc2VlZCBzIGlzIENSTi0KICAgICAgcGFpcmVkIGFjcm9zcyBhcm1zIChzaGFyZWQgZW52IHJlc2V0ICsgbmV0IGluaXQgKyByb2xsb3V0IHJhbmRvbW5lc3MpLgoKcmVsX2RlY2F5IG1hdGNoZXMgZXh0aW5jdC5hbmFseXplOiBmb3IgdGhlIG1hcmtlZCBiZXJyeSwgaXRzIG9wcG9ydHVuaXR5LXJhdGUgcmlzZQphY3Jvc3MgdGhlIHN3aXRjaCBNSU5VUyB0aGUgdW5tYXJrZWQgY29udHJvbCBiZXJyeSdzIGRyaWZ0LCBwZXIgc2VlZC4KIiIiCmltcG9ydCBjc3YgYXMgX2NzdgppbXBvcnQgbnVtcHkgYXMgbnAKCgpkZWYgX3N3aXRjaChyb3dzKToKICAgIGRlZiBpbnN0KHIpOgogICAgICAgIHJldHVybiAoZmxvYXQoclsiZW5mb3JjZSJdKSA+IDAuNSBhbmQgZmxvYXQoci5nZXQoImJvbnVzX29uIiwgMSkpID4gMC41CiAgICAgICAgICAgICAgICBhbmQgZmxvYXQoci5nZXQoInJlbW92YWxfb24iLCAxKSkgPiAwLjUpCiAgICByZXR1cm4gbWF4KChpbnQoclsidXBkYXRlIl0pIGZvciByIGluIHJvd3MgaWYgaW5zdChyKSksIGRlZmF1bHQ9LTEpCgoKZGVmIHNpbGx5X2RlY2F5cyhwYXRoLCBtYXJrZWRfYmVycnk9MSwgY29uZD1Ob25lLCB3aW5kb3c9MTAwKToKICAgICIiIlBlci1zZWVkIHJlbF9kZWNheSBmb3IgdGhlIG1hcmtlZCBiZXJyeSBpbiBpdHMgb3duIGNvbmRpdGlvbi4gUmV0dXJucyBucC5hcnJheQogICAgb3ZlciBzZWVkcyAoc29ydGVkKS4gRGVmYXVsdDogdGhlIHNpbGx5ICgxLCkgY29uZGl0aW9uLCBjb250cm9sID0gaGlnaGVzdCB1bm1hcmtlZC4iIiIKICAgIHJvd3MgPSBsaXN0KF9jc3YuRGljdFJlYWRlcihvcGVuKHBhdGgpKSkKICAgIG5idCA9IHN1bSgxIGZvciBrIGluIHJvd3NbMF0gaWYgay5zdGFydHN3aXRoKCJlYXQiKSkKICAgIGlmIGNvbmQgaXMgTm9uZToKICAgICAgICBjb25kID0gc3RyKG1hcmtlZF9iZXJyeSkgICAgICAgICAgICAgICAgICAgICAgICMgZS5nLiAiMSIgZm9yIHRoZSAoMSwpIHJ1bgogICAgdXBzID0gc29ydGVkKHtpbnQoclsidXBkYXRlIl0pIGZvciByIGluIHJvd3N9KQogICAgc3cgPSBfc3dpdGNoKHJvd3MpCiAgICB3MCA9IFt1IGZvciB1IGluIHVwcyBpZiB1IDw9IHN3XVstd2luZG93Ol0gICAgICAgICAjIGluc3RhbGwtZW5kCiAgICB3MSA9IFt1IGZvciB1IGluIHVwcyBpZiB1ID4gc3ddWy13aW5kb3c6XSAgICAgICAgICAjIGV4dGluY3Rpb24tZW5kCiAgICBjdHJsID0gbWF4KHQgZm9yIHQgaW4gcmFuZ2UobmJ0KSBpZiB0ICE9IG1hcmtlZF9iZXJyeSkKCiAgICBkZWYgb3BwKHNlZWQsIGksIHdpbik6CiAgICAgICAgcnMgPSBbciBmb3IgciBpbiByb3dzIGlmIHJbImNvbmRpdGlvbiJdID09IGNvbmQgYW5kIGludChyWyJzZWVkIl0pID09IHNlZWQKICAgICAgICAgICAgICBhbmQgaW50KHJbInVwZGF0ZSJdKSBpbiB3aW5dCiAgICAgICAgZSA9IHN1bShmbG9hdChyW2YiZWF0e2l9Il0pIGZvciByIGluIHJzKQogICAgICAgIGMgPSBzdW0oZmxvYXQocltmImVuY3tpfSJdKSBmb3IgciBpbiBycykKICAgICAgICByZXR1cm4gZSAvIG1heChjLCAxZS05KQoKICAgIHNlZWRzID0gc29ydGVkKHtpbnQoclsic2VlZCJdKSBmb3IgciBpbiByb3dzIGlmIHJbImNvbmRpdGlvbiJdID09IGNvbmR9KQogICAgb3V0ID0gW10KICAgIGZvciBzIGluIHNlZWRzOgogICAgICAgIGRtID0gb3BwKHMsIG1hcmtlZF9iZXJyeSwgdzEpIC0gb3BwKHMsIG1hcmtlZF9iZXJyeSwgdzApCiAgICAgICAgZHUgPSBvcHAocywgY3RybCwgdzEpIC0gb3BwKHMsIGN0cmwsIHcwKQogICAgICAgIG91dC5hcHBlbmQoZG0gLSBkdSkKICAgIHJldHVybiBucC5hcnJheShvdXQpCgoKZGVmIF9jaSh4KToKICAgIG0gPSB4Lm1lYW4oKTsgc2UgPSB4LnN0ZChkZG9mPTEpIC8gbnAuc3FydChsZW4oeCkpCiAgICByZXR1cm4gbSwgc2UsIG0gLSAxLjk2ICogc2UsIG0gKyAxLjk2ICogc2UKCgpkZWYgcmVwb3J0KHZpb2xhdG9yX2NzdiwgZW5mb3JjZXJfY3N2LCBlcHM9MC4wMiwgd2luZG93PTEwMCk6CiAgICAiIiJlcHMgPSBuZWdsaWdpYmxlLWVmZmVjdCBiYW5kIGZvciB0aGUgZXF1aXZhbGVuY2UgdGVzdCAodW5pdHMgb2YgcmVsX2RlY2F5KS4KICAgIERlZmF1bHQgMC4wMiB+IHRoZSBwb2lzb24gcGh5c2ljYWwtZmxvb3IgZGVjYXkgbWFnbml0dWRlOyB0dW5lIHBlciByZXZpZXdlci4iIiIKICAgIHZvID0gc2lsbHlfZGVjYXlzKHZpb2xhdG9yX2Nzdiwgd2luZG93PXdpbmRvdykKICAgIGVvID0gc2lsbHlfZGVjYXlzKGVuZm9yY2VyX2Nzdiwgd2luZG93PXdpbmRvdykKICAgIHByaW50KCI9IiAqIDc0KQogICAgcHJpbnQoZiJWQVJJQU5DRS1DT05UUk9MTEVEIFZJT0xBVE9SLU9OTFkgICh3aW5kb3c9bGFzdCB7d2luZG93fSwgZXBzPXtlcHN9KSIpCiAgICBwcmludChmIiAgdmlvbGF0b3Itb25seToge3Zpb2xhdG9yX2Nzdn0gIChuPXtsZW4odm8pfSBzZWVkcykiKQogICAgcHJpbnQoZiIgIGVuZm9yY2VyLW9ubHk6IHtlbmZvcmNlcl9jc3Z9ICAobj17bGVuKGVvKX0gc2VlZHMpIikKICAgIHByaW50KCItIiAqIDc0KQoKICAgIG0sIHNlLCBsbywgaGkgPSBfY2kodm8pCiAgICBwcmludCgiKDEpIEVRVUlWQUxFTkNFIG9uIHRoZSB2aW9sYXRvci1vbmx5IHNpbGx5IHJlbF9kZWNheSIpCiAgICBwcmludChmIiAgICBtZWFuIHttOisuNGZ9ICBTRSB7c2U6LjRmfSAgOTUlIENJIFt7bG86Ky40Zn0sIHtoaTorLjRmfV0iKQogICAgIyBUT1NUOiByZWplY3QgJ2VmZmVjdCA8PSAtZXBzJyBhbmQgJ2VmZmVjdCA+PSArZXBzJyA9PiBpbnNpZGUgdGhlIGJhbmQKICAgIHRfbG8gPSAobSAtICgtZXBzKSkgLyBzZSAgICAgICAgICAjIEgwOiA8PSAtZXBzIDsgd2FudCB0X2xvIGxhcmdlICsKICAgIHRfaGkgPSAoKGVwcykgLSBtKSAvIHNlICAgICAgICAgICAjIEgwOiA+PSArZXBzIDsgd2FudCB0X2hpIGxhcmdlICsKICAgIGluc2lkZSA9IChsbyA+IC1lcHMpIGFuZCAoaGkgPCBlcHMpCiAgICBwcmludChmIiAgICBUT1NUIHQoK2Vwcyk9e3RfaGk6Ky4yZn0gIHQoLWVwcyk9e3RfbG86Ky4yZn0gICIKICAgICAgICAgIGYiQ0kgd2l0aGluICsvLXtlcHN9OiB7aW5zaWRlfSAgLT4geydFUVVJVkFMRU5UIHRvIHplcm8gKHN1cHBvcnRlZCBudWxsKScgaWYgaW5zaWRlIGVsc2UgJ05PVCBlc3RhYmxpc2hlZCd9IikKCiAgICBwcmludCgiKDIpIFBBSVJFRCBJTlRFUkFDVElPTiAgZW5mb3JjZXItb25seSAtIHZpb2xhdG9yLW9ubHkgIChwZXItc2VlZCwgQ1JOKSIpCiAgICBpZiBsZW4odm8pICE9IGxlbihlbyk6CiAgICAgICAgcHJpbnQoZiIgICAgISEgc2VlZCBjb3VudHMgZGlmZmVyICh7bGVuKHZvKX0gdnMge2xlbihlbyl9KSAtPiBOT1QgQ1JOLXBhaXJlZDsgIgogICAgICAgICAgICAgIGYicmVwb3J0aW5nIFVOUEFJUkVEIChXZWxjaCkgY29udHJhc3QgaW5zdGVhZC4iKQogICAgICAgIG1kXyA9IGVvLm1lYW4oKSAtIHZvLm1lYW4oKQogICAgICAgIHNlXyA9IG5wLnNxcnQoZW8udmFyKGRkb2Y9MSkvbGVuKGVvKSArIHZvLnZhcihkZG9mPTEpL2xlbih2bykpCiAgICAgICAgcHJpbnQoZiIgICAgZGVsdGEge21kXzorLjRmfSAgbWFyZ2luIHttZF8vc2VfOisuMmZ9ICAodHdvLXNhbXBsZSkiKQogICAgZWxzZToKICAgICAgICBkID0gZW8gLSB2bwogICAgICAgIG1kXywgc2VkXywgZGxvLCBkaGkgPSBfY2koZCkKICAgICAgICBwcmludChmIiAgICBkZWx0YSB7bWRfOisuNGZ9ICBTRSB7c2VkXzouNGZ9ICBtYXJnaW4ge21kXy9zZWRfOisuMmZ9ICAiCiAgICAgICAgICAgICAgZiI5NSUgQ0kgW3tkbG86Ky40Zn0sIHtkaGk6Ky40Zn1dICB7KGQgPiAwKS5zdW0oKX0ve2xlbihkKX0gc2VlZHMgcG9zaXRpdmUiKQogICAgICAgIHByaW50KGYiICAgIC0+IGVuZm9yY2VyLW9ubHkgZGVjYXlzIHttZF86Ky40Zn0gTU9SRSB0aGFuIHZpb2xhdG9yLW9ubHksICIKICAgICAgICAgICAgICBmInsnQUxMIHNlZWRzJyBpZiAoZCA+IDApLmFsbCgpIGVsc2UgJ21vc3Qgc2VlZHMnfSBzYW1lIHNpZ24iKQogICAgcHJpbnQoIj0iICogNzQpCiAgICBwcmludCgiTWFudXNjcmlwdCBsaW5lOiB2aW9sYXRvci1vbmx5IHNpbGx5IHJlbF9kZWNheSBpcyBzdGF0aXN0aWNhbGx5IGVxdWl2YWxlbnQgdG8iKQogICAgcHJpbnQoInplcm8gKHdpdGhpbiArLy1lcHMpLCBhbmQgc2lnbmlmaWNhbnRseSBzbWFsbGVyIHRoYW4gdGhlIGVuZm9yY2VyLW9ubHkgZGVjYXkiKQogICAgcHJpbnQoImF0IG1hdGNoZWQgc2VlZHMgLT4gcmVtb3ZpbmcgdGhlIHZpb2xhdG9yJ3MgY29zdCBhbG9uZSBjaGFuZ2VzIG5vdGhpbmc7IHRoZSIpCiAgICBwcmludCgibm9ybSByZXN0cyBvbiB0aGUgZW5mb3JjZXIncyBpbmNlbnRpdmUuIikKICAgIHJldHVybiBkaWN0KHZpb2xhdG9yPXZvLCBlbmZvcmNlcj1lbykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaW1wb3J0IHN5cwogICAgYSA9IHN5cy5hcmd2WzE6XQogICAgcmVwb3J0KGFbMF0sIGFbMV0sIGVwcz1mbG9hdChhWzJdKSBpZiBsZW4oYSkgPiAyIGVsc2UgMC4wMikK',
}
for _n, _b in _SRC.items(): pathlib.Path(_n).write_bytes(base64.b64decode(_b))
print('wrote sources:', list(_SRC))


## 2. Run  (LONG cell -- expect `>>> RUN START`, per-chunk lines, then `>>> RUN DONE`)

In [ ]:
print('>>> RUN START', flush=True)
import run_sweep as R, extinct
ENV = R.env_variant(poison_delay=100, r_zap_bonus=8.75, episode_len=300,
                    zap_removal_steps=25, bonus_requires_mark=True,
                    c_zapped=2.0, n_berry_types=3, grid=22)
hp = dict(R.FAITHFUL_HP); hp['updates']=1600
extinct.run_extinction([(1,)], ENV, hp, n_install=1000, n_seeds=20,
                       vmap_seeds=True,  # chunked vmap, 5 seeds/graph (~30GB, fits 80GB)
                       out_csv='extinct3_vc_violatoronly.csv')
extinct.analyze('extinct3_vc_violatoronly.csv')
print('>>> RUN DONE', flush=True)
